# 03 - Model Training (PyTorch — Manual)
ConvLSTM spatiotemporal model: 3-month sequences -> 1-month fishing ground prediction.

**Changes from original Keras notebook:**
- `nn.Conv2d` replaced with manual im2col convolution via `F.unfold` + `torch.matmul`
- `nn.BatchNorm3d/2d` replaced with manual running-stat batch norm (pure tensor ops)
- `nn.Dropout` replaced with manual inverted-dropout via `torch.bernoulli`
- Output head 1×1 conv replaced with explicit channel-wise matmul
- `torch.optim.Adam` replaced with manual Adam (pure tensor ops, no NumPy)
- `binary_cross_entropy` replaced with `focal_loss` (fixes class imbalance on sparse fishing maps)
- `filters1=64, filters2=32` updated to `32, 16` (right-sized for ~90 training sequences, 2017-2024)
- Gradient clipping added before weight update
- LR reduction on plateau added to ManualAdam loop
- Class-imbalance diagnostic cell added after split
- Val F1 tracked per epoch; early stopping monitors val_f1 instead of val_loss
- date_model synced to 2017-01-01 to match preprocessing notebook
- norm_stats_json added to files dict

In [1]:
# CELL 1
!pip install -q torch xarray netCDF4

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 31.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 32.7 MB/s eta 0:00:00


In [2]:
# CELL 2
import xarray as xr
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import json, os

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

CONFIG = {
    'bbox_regional': {'lat_min': 0,  'lat_max': 30,  'lon_min': 110, 'lon_max': 140},
    'bbox_model':    {'lat_min': 10, 'lat_max': 20,  'lon_min': 114, 'lon_max': 120},
    'date_full':  {'start': '2014-01-01', 'end': '2024-12-31'},
    'date_model': {'start': '2017-01-01', 'end': '2024-12-31'},  # synced to preprocessing
    'data_dir': '/content/drive/MyDrive/fishing_project/',
    'files': {
        'physics_w_nc':    'cmems_mod_glo_phy_my_0.083deg_P1M-m_1779636039565.nc',
        'physics_ht_nc':   'cmems_mod_glo_phy_my_0.083deg_P1M-m_1779635380319.nc',
        'bgc_src_nc':      'cmems_mod_glo_bgc_my_0.25deg_P1M-m_1779635372583.nc',
        'physics_nc':      'physics_raw_region.nc',
        'bgc_nc':          'bgc_raw_region.nc',
        'ais_parquet':     'ais_raw_region.parquet',
        'ais_csv_gz':      'ais_raw_region.csv.gz',
        'ais_gridded_nc':  'ais_fishing_effort_gridded.nc',
        'preprocessed_nc': 'preprocessed_features.nc',
        'norm_stats_json': 'norm_stats.json',
        'model_pt':        'convlstm_model.pt',
        'best_model_pt':   'best_model.pt',
        'weights_npz':     'convlstm_weights.npz',
        'X_test_npy':      'X_test.npy',
        'y_test_npy':      'y_test.npy',
        'history_json':    'training_history.json',
        'summary_json':    'data_summary.json',
        'predictions_npy': 'predictions.npy',
        'eval_csv':        'evaluation_results.csv',
    },
    'ais_use_cols':          ['date', 'cell_ll_lat', 'cell_ll_lon', 'fishing_hours'],
    'physics_surface_depth': 0.49,
    'bgc_depth_range':       (0.51, 5.14),
    'norm_method':           'minmax',
    'resample_freq':         '1ME',
    'seq_len':      3,
    'pred_len':     1,
    'n_channels':   7,
    'train_frac':   0.70,
    'val_frac':     0.15,
    'epochs':       100,
    'batch_size':   4,    # small dataset; 4 gives ~15 gradient steps/epoch
    'patience':     15,
    'f1_threshold': 0.15,
}

DATA_DIR = CONFIG['data_dir']
f        = CONFIG['files']

print(f"PyTorch  : {torch.__version__}")
print(f"CUDA     : {torch.cuda.is_available()}")

Mounted at /content/drive
PyTorch  : 2.11.0+cpu
CUDA     : False


In [3]:
# CELL 3 — load preprocessed features
SEQ_LEN    = CONFIG['seq_len']
PRED_LEN   = CONFIG['pred_len']
N_CHANNELS = CONFIG['n_channels']

features = xr.open_dataset(DATA_DIR + f['preprocessed_nc'])
n_months = features.sizes['time']
n_lat    = features.sizes['latitude']
n_lon    = features.sizes['longitude']

print(f'Preprocessed features:')
print(f'  time={n_months}, latitude={n_lat}, longitude={n_lon}')
print(f'  Variables: {list(features.data_vars)}')

Preprocessed features:
  time=96, latitude=41, longitude=25
  Variables: ['chl', 'nppv', 'ssh', 'sst', 'uo', 'vo', 'fishing_effort']


In [4]:
# CELL 4 — stack channels
# Channel order matches Keras notebook: sst, ssh, vo, uo, chl, nppv, fishing_effort
X_stack = np.stack([
    features['sst'].values,
    features['ssh'].values,
    features['vo'].values,
    features['uo'].values,
    features['chl'].values,
    features['nppv'].values,
    features['fishing_effort'].values,
], axis=-1)  # (n_months, n_lat, n_lon, 7)

y_array = features['fishing_effort'].values  # (n_months, n_lat, n_lon)

X_stack = np.nan_to_num(X_stack, nan=0.0)
y_array = np.nan_to_num(y_array, nan=0.0)

print(f'X stack shape : {X_stack.shape}')
print(f'y array shape : {y_array.shape}')

X stack shape : (96, 41, 25, 7)
y array shape : (96, 41, 25)


In [5]:
# CELL 5 — sliding-window sequence generation
def create_sequences(X, y, seq_len, pred_len):
    X_seq, y_seq = [], []
    for i in range(len(X) - seq_len - pred_len + 1):
        X_seq.append(X[i : i + seq_len])
        y_seq.append(y[i + seq_len : i + seq_len + pred_len])
    return np.array(X_seq), np.array(y_seq)

X_seq, y_seq = create_sequences(X_stack, y_array, SEQ_LEN, PRED_LEN)
print(f'Sequences : {X_seq.shape} -> {y_seq.shape}')
# Expected with 2017-2024 (~96 months): (92, 3, n_lat, n_lon, 7) -> (92, 1, n_lat, n_lon)

Sequences : (93, 3, 41, 25, 7) -> (93, 1, 41, 25)


In [6]:
# CELL 6 — chronological train / val / test split (70 / 15 / 15)
train_size = int(CONFIG['train_frac'] * len(X_seq))
val_size   = int(CONFIG['val_frac']   * len(X_seq))

X_train = X_seq[:train_size]
y_train = y_seq[:train_size]
X_val   = X_seq[train_size : train_size + val_size]
y_val   = y_seq[train_size : train_size + val_size]
X_test  = X_seq[train_size + val_size :]
y_test  = y_seq[train_size + val_size :]

print(f'Train : {X_train.shape}')
print(f'Val   : {X_val.shape}')
print(f'Test  : {X_test.shape}')

np.save(DATA_DIR + f['X_test_npy'], X_test)
print(f'Saved X_test -> {f["X_test_npy"]}')
print(f'X_test layout — should be (N, T, H, W, C): {X_test.shape}')

Train : (65, 3, 41, 25, 7)
Val   : (13, 3, 41, 25, 7)
Test  : (15, 3, 41, 25, 7)
Saved X_test -> X_test.npy
X_test layout — should be (N, T, H, W, C): (15, 3, 41, 25, 7)


In [7]:
# CELL 7 — reshape y to (N, H, W, 1) matching Keras convention
y_train      = y_train.squeeze(axis=1)[..., np.newaxis]   # (N_train, n_lat, n_lon, 1)
y_val        = y_val.squeeze(axis=1)[..., np.newaxis]
y_test_model = y_test.squeeze(axis=1)[..., np.newaxis]

np.save(DATA_DIR + f['y_test_npy'], y_test_model)
print(f'Saved y_test -> {f["y_test_npy"]}  shape={y_test_model.shape}')
print(f'y_train : {y_train.shape}')
print(f'y_val   : {y_val.shape}')

Saved y_test -> y_test.npy  shape=(15, 41, 25, 1)
y_train : (65, 41, 25, 1)
y_val   : (13, 41, 25, 1)


In [8]:
# CELL 7b — class-imbalance diagnostic
#
# Fishing cells are a tiny fraction of the grid. BCE converges to
# predicting all-zeros and reports low loss without ever finding fishing.
# This cell measures the positive ratio so focal_loss alpha can be set
# appropriately, and confirms the f1_threshold makes sense.

threshold = CONFIG['f1_threshold']

pos_train = (y_train > threshold).mean()
pos_val   = (y_val   > threshold).mean()

print(f'Positive cell ratio (threshold={threshold}):')
print(f'  Train : {pos_train:.4f}  ({pos_train*100:.2f}% of cells are fishing)')
print(f'  Val   : {pos_val:.4f}  ({pos_val*100:.2f}% of cells are fishing)')
print()

# alpha=0.75 is a robust default for sparse fishing maps.
# Increase toward 0.90 if the model predicts all-zero after 10 epochs.
# Decrease toward 0.60 if the model predicts fishing everywhere.
# DO NOT use 1 - pos_ratio; that's far too extreme and causes divergence.
FOCAL_ALPHA = 0.75
FOCAL_GAMMA = 2.0

print(f'pos_train={pos_train:.4f} — using FOCAL_ALPHA={FOCAL_ALPHA}, FOCAL_GAMMA={FOCAL_GAMMA}')
print(f'(Adjust alpha up if model ignores positives, down if it predicts fishing everywhere)')

Positive cell ratio (threshold=0.15):
  Train : 0.0244  (2.44% of cells are fishing)
  Val   : 0.2473  (24.73% of cells are fishing)

pos_train=0.0244 — using FOCAL_ALPHA=0.75, FOCAL_GAMMA=2.0
(Adjust alpha up if model ignores positives, down if it predicts fishing everywhere)


In [9]:
# CELL 8 — ConvLSTMCell with manual im2col convolution
#
# nn.Conv2d replaced with:
#   - nn.Parameter for raw weight tensors (shape [C_out, C_in, kH, kW])
#   - manual_conv2d() using F.unfold (im2col) + torch.matmul + reshape
#
# HOW F.unfold IMPLEMENTS CONVOLUTION:
#   Input (B, C_in, H, W)
#   F.unfold extracts every kH×kW patch at every (h,w) position
#     -> (B, C_in*kH*kW, H*W)   [the im2col matrix]
#   Weight flattened to (C_out, C_in*kH*kW)
#   matmul -> (B, C_out, H*W) -> reshape -> (B, C_out, H, W)
#   Mathematically identical to sliding-window convolution.

def manual_conv2d(x, W, b, kernel_size, padding):
    """
    Manual 2D convolution via im2col (F.unfold) + matmul.

    x : (B, C_in, H, W)
    W : (C_out, C_in, kH, kW)  — raw weight parameter
    b : (C_out,)               — bias (or None)

    Returns (B, C_out, H, W)
    """
    # Use W_spatial to avoid shadowing the weight parameter W
    B, C_in, H, W_spatial = x.shape
    C_out = W.shape[0]

    # Step 1: extract sliding patches — im2col
    # unfolded: (B, C_in * kH * kW, H * W)
    unfolded = F.unfold(x, kernel_size=kernel_size, padding=padding)

    # Step 2: flatten kernel weights to (C_out, C_in * kH * kW)
    W_flat = W.view(C_out, -1)

    # Step 3: matmul — (C_out, col) x (col, H*W) per batch item
    out = unfolded.transpose(1, 2).matmul(W_flat.t())  # (B, H*W, C_out)

    # Step 4: reshape to spatial output (using W_spatial)
    out = out.transpose(1, 2).view(B, C_out, H, W_spatial)    # (B, C_out, H, W_spatial)

    if b is not None:
        out = out + b.view(1, C_out, 1, 1)             # broadcast bias

    return out

class ConvLSTMCell(nn.Module):
    """
    Single-step ConvLSTM cell. Convolutions done via manual im2col.

    Gate layout (channel axis): [i | f | g | o] each of size hidden_channels.

    Shapes:
      x_t    : (B, in_channels,     H, W)
      h_prev : (B, hidden_channels, H, W)
      c_prev : (B, hidden_channels, H, W)
    """
    def __init__(self, in_channels, hidden_channels, kernel_size=3):
        super().__init__()
        self.hidden_channels = hidden_channels
        self.kernel_size     = kernel_size
        self.padding         = kernel_size // 2  # 'same' padding

        # Xavier uniform init: limit = sqrt(6 / (fan_in + fan_out))
        fan_in_x  = in_channels      * kernel_size * kernel_size
        fan_out_x = 4 * hidden_channels * kernel_size * kernel_size
        lim_x = (6.0 / (fan_in_x + fan_out_x)) ** 0.5
        self.Wx = nn.Parameter(
            torch.empty(4 * hidden_channels, in_channels, kernel_size, kernel_size)
            .uniform_(-lim_x, lim_x)
        )

        fan_in_h  = hidden_channels     * kernel_size * kernel_size
        fan_out_h = 4 * hidden_channels * kernel_size * kernel_size
        lim_h = (6.0 / (fan_in_h + fan_out_h)) ** 0.5
        self.Wh = nn.Parameter(
            torch.empty(4 * hidden_channels, hidden_channels, kernel_size, kernel_size)
            .uniform_(-lim_h, lim_h)
        )

        # Bias: forget gate slice [hidden:2*hidden] init to +1 to prevent
        # forgetting everything at the start of training.
        b_init = torch.zeros(4 * hidden_channels)
        b_init[hidden_channels : 2 * hidden_channels] = 1.0
        self.b = nn.Parameter(b_init)

    def forward(self, x_t, h_prev, c_prev):
        # gates: (B, 4*hidden, H, W)
        gates = (
            manual_conv2d(x_t,    self.Wx, None, self.kernel_size, self.padding)
            + manual_conv2d(h_prev, self.Wh, None, self.kernel_size, self.padding)
            + self.b.view(1, -1, 1, 1)
        )

        i_g, f_g, g_g, o_g = gates.chunk(4, dim=1)

        i = torch.sigmoid(i_g)   # input  gate
        f = torch.sigmoid(f_g)   # forget gate
        g = torch.tanh(g_g)      # cell   gate
        o = torch.sigmoid(o_g)   # output gate

        c_t = f * c_prev + i * g
        h_t = o * torch.tanh(c_t)

        return h_t, c_t


print('ConvLSTMCell (manual im2col) defined.')

ConvLSTMCell (manual im2col) defined.


In [10]:
# CELL 9 — ConvLSTMLayer + ConvLSTMModel
#
# WHAT CHANGED vs original Keras:
#   1. filters1 64->32, filters2 32->16
#      ~90 training sequences (2017-2024); 32/16 is appropriate capacity
#      and stays well under 200K parameters.
#
#   2. nn.BatchNorm3d/2d -> manual_batch_norm()
#      Reduces over all dims except last (channel), maintains running
#      stats as non-grad nn.Parameter. eps=1e-4 for stability with
#      small batches (batch_size=4).
#
#   3. nn.Dropout -> manual_dropout()
#      Bernoulli mask, inverted scaling.
#
#   4. nn.Conv2d(filters2, 1, 1) -> manual 1x1 conv via matmul.

FILTERS1 = 32
FILTERS2 = 16


def manual_batch_norm(x, gamma, beta, running_mean, running_var,
                      training, eps=1e-4, momentum=0.1):
    """
    Manual batch normalisation. Channel dim must be LAST in x.

    MATH:
      x_hat = (x - mean) / sqrt(var + eps)
      y     = gamma * x_hat + beta

    eps=1e-4 (not 1e-5) for stability with small batch sizes.
    """
    if training:
        reduce_dims = tuple(range(x.dim() - 1))  # all dims except last
        mean = x.mean(dim=reduce_dims, keepdim=True)
        var  = x.var( dim=reduce_dims, keepdim=True, unbiased=False)

        with torch.no_grad():
            running_mean.copy_(
                (1 - momentum) * running_mean + momentum * mean.squeeze()
            )
            running_var.copy_(
                (1 - momentum) * running_var  + momentum * var.squeeze()
            )
    else:
        shape = [1] * x.dim()
        shape[-1] = -1
        mean = running_mean.view(shape)
        var  = running_var.view(shape)

    x_hat = (x - mean) / torch.sqrt(var + eps)
    return gamma * x_hat + beta


def manual_dropout(x, rate, training):
    """
    Inverted dropout. Survivors scaled by 1/(1-rate) during training
    so eval pass requires no scaling.
    """
    if not training or rate == 0.0:
        return x
    keep = torch.bernoulli(torch.full(x.shape, 1.0 - rate,
                                      device=x.device, dtype=x.dtype))
    return x * keep / (1.0 - rate)


class ConvLSTMLayer(nn.Module):
    """Unrolls ConvLSTMCell over a (B, T, C, H, W) sequence.
    h and c reset to zero at the start of each sequence (stateless across batches).
    This is correct: each (X_seq[i], y_seq[i]) pair is independent.
    """
    def __init__(self, in_channels, hidden_channels, kernel_size=3,
                 return_sequences=True):
        super().__init__()
        self.cell             = ConvLSTMCell(in_channels, hidden_channels, kernel_size)
        self.hidden_channels  = hidden_channels
        self.return_sequences = return_sequences

    def forward(self, x_seq):
        B, T, C, H, W = x_seq.shape
        h = torch.zeros(B, self.hidden_channels, H, W,
                        device=x_seq.device, dtype=x_seq.dtype)
        c = torch.zeros(B, self.hidden_channels, H, W,
                        device=x_seq.device, dtype=x_seq.dtype)
        outputs = []
        for t in range(T):
            h, c = self.cell(x_seq[:, t], h, c)
            outputs.append(h)
        if self.return_sequences:
            return torch.stack(outputs, dim=1)  # (B, T, hidden, H, W)
        return outputs[-1]                      # (B, hidden, H, W)


class ConvLSTMModel(nn.Module):
    """
    2-layer ConvLSTM with manual BN, manual dropout, manual 1x1 output conv.

    Input  : (B, T=3, C=7,  H=n_lat, W=n_lon)
    Output : (B, 1,         H=n_lat, W=n_lon)  in (0,1) via sigmoid

    Architecture:
      ConvLSTMLayer(7  -> 32, return_sequences=True)
      BatchNorm (manual, channel dim = 32)
      Dropout   (manual, rate=0.2)
      ConvLSTMLayer(32 -> 16, return_sequences=False)
      BatchNorm (manual, channel dim = 16)
      Dropout   (manual, rate=0.2)
      1x1 conv  (manual matmul, 16 -> 1)
      sigmoid
    """
    def __init__(self, n_channels=7, filters1=32, filters2=16,
                 kernel_size=3, dropout_rate=0.2):
        super().__init__()
        self.dropout_rate = dropout_rate
        self.filters1     = filters1
        self.filters2     = filters2

        self.layer1 = ConvLSTMLayer(n_channels, filters1, kernel_size,
                                    return_sequences=True)
        self.layer2 = ConvLSTMLayer(filters1,   filters2, kernel_size,
                                    return_sequences=False)

        # BN params for layer1 output (B, T, filters1, H, W)
        # Permuted to (B, T, H, W, filters1) before calling manual_batch_norm
        self.bn1_gamma = nn.Parameter(torch.ones(filters1))
        self.bn1_beta  = nn.Parameter(torch.zeros(filters1))
        self.bn1_rmean = nn.Parameter(torch.zeros(filters1), requires_grad=False)
        self.bn1_rvar  = nn.Parameter(torch.ones(filters1),  requires_grad=False)

        # BN params for layer2 output (B, filters2, H, W)
        self.bn2_gamma = nn.Parameter(torch.ones(filters2))
        self.bn2_beta  = nn.Parameter(torch.zeros(filters2))
        self.bn2_rmean = nn.Parameter(torch.zeros(filters2), requires_grad=False)
        self.bn2_rvar  = nn.Parameter(torch.ones(filters2),  requires_grad=False)

        # Manual 1x1 output conv: (filters2 -> 1)
        lim = (6.0 / (filters2 + 1)) ** 0.5
        self.W_out = nn.Parameter(torch.empty(1, filters2).uniform_(-lim, lim))
        self.b_out = nn.Parameter(torch.zeros(1))

    def forward(self, x, training=None):
        """
        x        : (B, T, C, H, W)
        training : bool — if None, inferred from self.training
        """
        if training is None:
            training = self.training

        # Block 1
        out = self.layer1(x)                # (B, T, filters1, H, W)
        out = out.permute(0, 1, 3, 4, 2)   # (B, T, H, W, filters1) — channel last for BN
        out = manual_batch_norm(
            out, self.bn1_gamma, self.bn1_beta,
            self.bn1_rmean, self.bn1_rvar, training
        )
        out = out.permute(0, 1, 4, 2, 3)   # (B, T, filters1, H, W)
        out = manual_dropout(out, self.dropout_rate, training)

        # Block 2
        out = self.layer2(out)              # (B, filters2, H, W)
        out = out.permute(0, 2, 3, 1)      # (B, H, W, filters2) — channel last for BN
        out = manual_batch_norm(
            out, self.bn2_gamma, self.bn2_beta,
            self.bn2_rmean, self.bn2_rvar, training
        )
        out = out.permute(0, 3, 1, 2)      # (B, filters2, H, W)
        out = manual_dropout(out, self.dropout_rate, training)

        # Output head: manual 1x1 conv
        out = out.permute(0, 2, 3, 1)                          # (B, H, W, filters2)
        out = out.matmul(self.W_out.t()) + self.b_out          # (B, H, W, 1)
        out = out.permute(0, 3, 1, 2)                          # (B, 1, H, W)

        return torch.sigmoid(out)

    def count_params(self):
        return sum(p.numel() for p in self.parameters() if p.requires_grad)


print('ConvLSTMLayer and ConvLSTMModel (manual BN, dropout, 1x1 conv) defined.')

ConvLSTMLayer and ConvLSTMModel (manual BN, dropout, 1x1 conv) defined.


In [11]:
# CELL 10 — loss functions, LR schedule constants, ManualAdam, model instantiation

# ── Loss functions ────────────────────────────────────────────────────────

def binary_cross_entropy(pred, target, eps=1e-7):
    """Standard BCE — kept for ablation reference."""
    pred = pred.clamp(eps, 1.0 - eps)
    return -(target * pred.log() + (1.0 - target) * (1.0 - pred).log()).mean()


def focal_loss(pred, target, alpha=FOCAL_ALPHA, gamma=FOCAL_GAMMA, eps=1e-7):
    """
    Focal loss for continuous targets (Lin et al. 2017 — adapted).

    Targets are soft values in [0,1] (normalized fishing hours), not hard binary.
    p_t is computed as a soft blend rather than a hard 0.5 threshold:
      p_t = pred * target + (1 - pred) * (1 - target)
    This means pixels with target=0.3 are not treated as pure negatives.

    Modulating factor (1-p_t)^gamma suppresses easy well-classified pixels
    (vast empty ocean) and focuses training on hard fishing-ground boundaries.

    alpha upweights the positive class without being so extreme it causes
    the model to predict fishing everywhere.

    pred, target: (B, 1, H, W) in [0, 1]
    """
    pred  = pred.clamp(eps, 1.0 - eps)
    bce   = -(target * pred.log() + (1.0 - target) * (1.0 - pred).log())
    p_t   = pred * target + (1.0 - pred) * (1.0 - target)
    mod   = (1.0 - p_t) ** gamma
    return (alpha * mod * bce).mean()


# ── LR schedule constants (module-level so training loop can see them) ────
LR_PATIENCE = 7    # epochs without val_f1 improvement before reducing LR
LR_FACTOR   = 0.5  # multiply lr by this on plateau
LR_MIN      = 1e-5 # lr floor


# ── Manual Adam ──────────────────────────────────────────────────────────

class ManualAdam:
    """
    Adam optimiser (Kingma & Ba 2015) — pure PyTorch tensor ops, no NumPy.

    UPDATE EQUATIONS per parameter θ:
      m_t  = β1*m_(t-1) + (1-β1)*g_t          [1st moment]
      v_t  = β2*v_(t-1) + (1-β2)*g_t²          [2nd moment]
      m̂_t  = m_t / (1 - β1^t)                  [bias correction]
      v̂_t  = v_t / (1 - β2^t)
      θ_t  = θ_(t-1) - lr * m̂_t / (sqrt(v̂_t) + ε)

    lr is a plain Python float so the training loop can update it directly
    via optimizer.lr = new_value.
    """
    def __init__(self, params, lr=1e-3, beta1=0.9, beta2=0.999, eps=1e-8):
        self.params = list(params)
        self.lr     = lr
        self.beta1  = beta1
        self.beta2  = beta2
        self.eps    = eps
        self.t      = 0
        self.m = [torch.zeros_like(p.data) for p in self.params]
        self.v = [torch.zeros_like(p.data) for p in self.params]

    def zero_grad(self):
        for p in self.params:
            if p.grad is not None:
                p.grad.zero_()

    def step(self):
        self.t += 1
        bias_c1 = 1.0 - self.beta1 ** self.t
        bias_c2 = 1.0 - self.beta2 ** self.t
        for p, m, v in zip(self.params, self.m, self.v):
            if p.grad is None:
                continue
            g = p.grad.data
            m.mul_(self.beta1).add_(g, alpha=1.0 - self.beta1)
            v.mul_(self.beta2).addcmul_(g, g, value=1.0 - self.beta2)
            m_hat = m / bias_c1
            v_hat = v / bias_c2
            p.data.addcdiv_(m_hat, v_hat.sqrt().add_(self.eps), value=-self.lr)


# ── Model + optimiser ────────────────────────────────────────────────────

DEVICE    = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
GRAD_CLIP = 1.0

model = ConvLSTMModel(
    n_channels   = N_CHANNELS,
    filters1     = FILTERS1,
    filters2     = FILTERS2,
    kernel_size  = 3,
    dropout_rate = 0.2,
).to(DEVICE)

optimizer = ManualAdam(
    (p for p in model.parameters() if p.requires_grad),
    lr=1e-3
)

print(f'Device     : {DEVICE}')
print(f'Parameters : {model.count_params():,}')

# ── Shape smoke-test ─────────────────────────────────────────────────────
print('\nShape smoke-test (random batch=2):')
model.eval()
with torch.no_grad():
    dummy = torch.rand(2, SEQ_LEN, N_CHANNELS, n_lat, n_lon)  # rand, not zeros
    out   = model(dummy)
    print(f'  Input  : {tuple(dummy.shape)}')
    print(f'  Output : {tuple(out.shape)}')
    assert out.shape == (2, 1, n_lat, n_lon), 'Shape mismatch!'
    assert out.min() >= 0 and out.max() <= 1, 'Sigmoid range violated!'
    assert not torch.isnan(out).any(), 'NaN in output!'
    print('  OK — output in (0,1), no NaNs')
model.train()

Device     : cpu
Parameters : 72,881

Shape smoke-test (random batch=2):
  Input  : (2, 3, 7, 41, 25)
  Output : (2, 1, 41, 25)
  OK — output in (0,1), no NaNs


ConvLSTMModel(
  (layer1): ConvLSTMLayer(
    (cell): ConvLSTMCell()
  )
  (layer2): ConvLSTMLayer(
    (cell): ConvLSTMCell()
  )
)

In [12]:
# CELL 11 — DataLoaders + training loop
#
# Key changes vs original Keras:
#   - focal_loss instead of binary_crossentropy
#   - ManualAdam with LR reduction on plateau
#   - Gradient clipping (GRAD_CLIP=1.0)
#   - Val F1 computed each epoch; early stopping monitors val_f1 not val_loss
#   - model.train() / model.eval() routes through manual BN/dropout via self.training

from torch.utils.data import TensorDataset, DataLoader

EPOCHS     = CONFIG['epochs']
BATCH_SIZE = CONFIG['batch_size']
PATIENCE   = CONFIG['patience']


def make_loader(X_np, y_np, batch_size, shuffle=False):
    # Transpose: TF layout (N, T, H, W, C) -> PyTorch (N, T, C, H, W)
    #            TF layout (N, H, W, 1)    -> PyTorch (N, 1, H, W)
    X_pt = torch.from_numpy(X_np.transpose(0, 1, 4, 2, 3)).float()
    y_pt = torch.from_numpy(y_np.transpose(0, 3, 1, 2)).float()
    return DataLoader(TensorDataset(X_pt, y_pt),
                      batch_size=batch_size, shuffle=shuffle)


train_loader = make_loader(X_train, y_train, BATCH_SIZE, shuffle=True)
val_loader   = make_loader(X_val,   y_val,   BATCH_SIZE, shuffle=False)

print(f'Train batches : {len(train_loader)}')
print(f'Val   batches : {len(val_loader)}')
print(f'Focal loss    : alpha={FOCAL_ALPHA:.3f}, gamma={FOCAL_GAMMA}')
print(f'Grad clip     : {GRAD_CLIP}')
print(f'Starting training — up to {EPOCHS} epochs, patience={PATIENCE} ...')
print()

# ── Training state ───────────────────────────────────────────────────────
best_val_f1       = -1.0          # monitor F1, not val_loss
patience_count    = 0
lr_patience_count = 0             # separate counter for LR reduction
best_state        = None
history           = {'loss': [], 'mae': [], 'val_loss': [], 'val_mae': [], 'val_f1': []}


for epoch in range(1, EPOCHS + 1):

    # ══ Train ════════════════════════════════════════════════════════════
    model.train()
    t_losses, t_maes = [], []

    for xb, yb in train_loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)

        optimizer.zero_grad()
        preds = model(xb)
        loss  = focal_loss(preds, yb)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            [p for p in model.parameters() if p.grad is not None],
            max_norm=GRAD_CLIP
        )
        optimizer.step()

        t_losses.append(loss.item())
        t_maes.append((preds.detach() - yb).abs().mean().item())

    # ══ Validate ═════════════════════════════════════════════════════════
    model.eval()
    v_losses, v_maes = [], []
    all_preds, all_targets = [], []

    with torch.no_grad():
        for xb, yb in val_loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            preds  = model(xb)
            v_losses.append(focal_loss(preds, yb).item())
            v_maes.append((preds - yb).abs().mean().item())
            all_preds.append(preds.cpu())
            all_targets.append(yb.cpu())

    # ── Val F1 (binarized at f1_threshold) ───────────────────────────────
    vp     = torch.cat(all_preds)
    vt     = torch.cat(all_targets)
    thresh = CONFIG['f1_threshold']
    vp_bin = (vp > thresh).float()
    vt_bin = (vt > thresh).float()
    tp     = (vp_bin * vt_bin).sum().item()
    fp     = (vp_bin * (1 - vt_bin)).sum().item()
    fn     = ((1 - vp_bin) * vt_bin).sum().item()
    val_f1 = tp / (tp + 0.5 * (fp + fn) + 1e-8)

    tl = float(np.mean(t_losses))
    tm = float(np.mean(t_maes))
    vl = float(np.mean(v_losses))
    vm = float(np.mean(v_maes))
    history['loss'].append(tl)
    history['mae'].append(tm)
    history['val_loss'].append(vl)
    history['val_mae'].append(vm)
    history['val_f1'].append(val_f1)

    print(f'Epoch {epoch:3d}/{EPOCHS}  '
          f'loss={tl:.4f}  mae={tm:.4f}  '
          f'val_loss={vl:.4f}  val_mae={vm:.4f}  val_f1={val_f1:.4f}')

    # ══ Checkpoint + early stopping (monitored on val_f1) ════════════════
    if val_f1 > best_val_f1:
        best_val_f1       = val_f1
        patience_count    = 0
        lr_patience_count = 0
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
        print(f'           ✓ val_f1 improved to {best_val_f1:.4f} -> checkpoint saved')
    else:
        patience_count    += 1
        lr_patience_count += 1
        if lr_patience_count >= LR_PATIENCE:
            optimizer.lr      = max(optimizer.lr * LR_FACTOR, LR_MIN)
            lr_patience_count = 0
            print(f'           ↓ LR reduced to {optimizer.lr:.2e}')
        if patience_count >= PATIENCE:
            print(f'Early stopping at epoch {epoch}.')
            break

model.load_state_dict(best_state)
best_epoch = int(np.argmax(history['val_f1'])) + 1
print(f'\nTraining complete — best val_f1={best_val_f1:.4f} at epoch {best_epoch}')

Train batches : 17
Val   batches : 4
Focal loss    : alpha=0.750, gamma=2.0
Grad clip     : 1.0
Starting training — up to 100 epochs, patience=15 ...

Epoch   1/100  loss=0.2313  mae=0.4846  val_loss=0.1014  val_mae=0.3764  val_f1=0.3965
           ✓ val_f1 improved to 0.3965 -> checkpoint saved
Epoch   2/100  loss=0.1620  mae=0.4630  val_loss=0.0951  val_mae=0.3634  val_f1=0.3965
Epoch   3/100  loss=0.1345  mae=0.4393  val_loss=0.0971  val_mae=0.3611  val_f1=0.3965
Epoch   4/100  loss=0.1150  mae=0.4250  val_loss=0.0991  val_mae=0.3608  val_f1=0.3969
           ✓ val_f1 improved to 0.3969 -> checkpoint saved
Epoch   5/100  loss=0.0995  mae=0.4088  val_loss=0.0960  val_mae=0.3563  val_f1=0.3974
           ✓ val_f1 improved to 0.3974 -> checkpoint saved
Epoch   6/100  loss=0.0871  mae=0.3940  val_loss=0.0829  val_mae=0.3353  val_f1=0.3986
           ✓ val_f1 improved to 0.3986 -> checkpoint saved
Epoch   7/100  loss=0.0772  mae=0.3767  val_loss=0.0753  val_mae=0.3192  val_f1=0.4046
    

In [13]:
# CELL 12 — save model, history, summary, and NumPy weights

# ── Model checkpoints ────────────────────────────────────────────────────
torch.save(model.state_dict(), DATA_DIR + f['model_pt'])
torch.save(model.state_dict(), DATA_DIR + f['best_model_pt'])
print(f'Saved model      -> {f["model_pt"]}')
print(f'Saved best model -> {f["best_model_pt"]}')

# ── Training history ─────────────────────────────────────────────────────
history_dict = {k: [float(v) for v in vals] for k, vals in history.items()}
with open(DATA_DIR + f['history_json'], 'w') as fh:
    json.dump(history_dict, fh, indent=2)
print(f'Saved history    -> {f["history_json"]}')

# ── Data + run summary ───────────────────────────────────────────────────
bm = CONFIG['bbox_model']
dm = CONFIG['date_model']
summary = {
    'date_range':    f"{dm['start']} to {dm['end']}",
    'n_months':      int(n_months),
    'n_lat':         int(n_lat),
    'n_lon':         int(n_lon),
    'n_channels':    int(N_CHANNELS),
    'seq_len':       int(SEQ_LEN),
    'pred_len':      int(PRED_LEN),
    'bbox_model':    bm,
    'n_train':       int(X_train.shape[0]),
    'n_val':         int(X_val.shape[0]),
    'n_test':        int(X_test.shape[0]),
    'best_epoch':    best_epoch,
    'best_val_f1':   float(best_val_f1),
    'filters1':      FILTERS1,
    'filters2':      FILTERS2,
    'focal_alpha':   FOCAL_ALPHA,
    'focal_gamma':   FOCAL_GAMMA,
    'grad_clip':     GRAD_CLIP,
    'batch_size':    BATCH_SIZE,
    'lr_final':      optimizer.lr,
}
with open(DATA_DIR + f['summary_json'], 'w') as fh:
    json.dump(summary, fh, indent=2)
print(f'Saved summary    -> {f["summary_json"]}')

# ── Export weights to NumPy .npz for 04_evaluation.ipynb ─────────────────
# PyTorch conv weight layout: (C_out, C_in, kH, kW)
# NumPy im2col layout:        (kH*kW*C_in, C_out)
# pt_to_np_conv() transposes and reshapes between the two.

sd = model.state_dict()

def t(key):
    return sd[key].cpu().numpy().astype(np.float32)

def pt_to_np_conv(w_pt):
    """(C_out, C_in, kH, kW) -> (kH*kW*C_in, C_out)"""
    C_out, C_in, kH, kW = w_pt.shape
    return w_pt.transpose(1, 2, 3, 0).reshape(C_in * kH * kW, C_out)

F1 = FILTERS1
F2 = FILTERS2

l1_Wx_pt = t('layer1.cell.Wx')
l1_Wh_pt = t('layer1.cell.Wh')
l2_Wx_pt = t('layer2.cell.Wx')
l2_Wh_pt = t('layer2.cell.Wh')

np.savez(
    DATA_DIR + f['weights_npz'],
    l1_Wx     = pt_to_np_conv(l1_Wx_pt),
    l1_Wh     = pt_to_np_conv(l1_Wh_pt),
    l1_b      = t('layer1.cell.b'),
    bn1_gamma = t('bn1_gamma'),
    bn1_beta  = t('bn1_beta'),
    bn1_rmean = t('bn1_rmean'),
    bn1_rvar  = t('bn1_rvar'),
    l2_Wx     = pt_to_np_conv(l2_Wx_pt),
    l2_Wh     = pt_to_np_conv(l2_Wh_pt),
    l2_b      = t('layer2.cell.b'),
    bn2_gamma = t('bn2_gamma'),
    bn2_beta  = t('bn2_beta'),
    bn2_rmean = t('bn2_rmean'),
    bn2_rvar  = t('bn2_rvar'),
    W_out     = t('W_out').T,   # (F2, 1)
    b_out     = t('b_out'),
)
print(f'Saved NumPy weights -> {f["weights_npz"]}')
print('Notebook 03 complete. Run 04_evaluation.ipynb next.')

Saved model      -> convlstm_model.pt
Saved best model -> best_model.pt
Saved history    -> training_history.json
Saved summary    -> data_summary.json
Saved NumPy weights -> convlstm_weights.npz
Notebook 03 complete. Run 04_evaluation.ipynb next.
